# Neuro-Symbolic Reasoning Datasets: CLUTRR, RuleTaker, ProofWriter

This notebook demonstrates loading and exploring three benchmark datasets for logical reasoning:
- **CLUTRR**: Compositional language understanding via kinship multi-hop reasoning
- **RuleTaker**: Rule-based logical entailment from context
- **ProofWriter**: Logical entailment with explicit proof traces

All datasets are standardized to a unified schema for neuro-symbolic reasoning experiments that convert unstructured text to first-order logic (FOL) representations.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-57cbed-demand-driven-fact-extraction-for-neuro/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini_demo_data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

Tunable parameters for this demo. Start with minimal values to prototype, then scale up as needed.

In [ ]:
# Configuration parameters for the demo
MAX_EXAMPLES_PER_DATASET = 3  # Start small: use all 3 examples we have
DISPLAY_INPUT_TRUNCATE_LENGTH = 200  # Truncate long inputs for readable display
VERBOSE = True  # Print detailed info during processing

## Processing: Extract and Standardize Examples

Load all examples from each dataset and standardize to the unified schema:
- `input`: The problem statement (story/context + query/statement)
- `output`: The expected answer
- `metadata_*`: Task-specific metadata (hop depth, task type, etc.)

In [ ]:
# Extract all examples and organize by dataset
all_examples = {}

for dataset_obj in data['datasets']:
    dataset_name = dataset_obj['dataset']
    examples = dataset_obj['examples']
    
    # Limit to MAX_EXAMPLES_PER_DATASET
    limited_examples = examples[:MAX_EXAMPLES_PER_DATASET]
    all_examples[dataset_name] = limited_examples
    
    if VERBOSE:
        print(f"\n{dataset_name}: {len(limited_examples)} examples loaded")
        for i, ex in enumerate(limited_examples, 1):
            print(f"  Example {i}: output={ex['output']}")

## Analysis: Schema Validation and Metadata Extraction

Verify all examples conform to the unified schema and extract metadata statistics.

In [ ]:
# Check schema: all examples should have these fields
required_fields = {'input', 'output', 'metadata_task_type'}

schema_valid = True
for dataset_name, examples in all_examples.items():
    for i, ex in enumerate(examples):
        if not required_fields.issubset(ex.keys()):
            print(f"  ERROR in {dataset_name} example {i}: missing fields")
            schema_valid = False

if schema_valid:
    print("✓ All examples conform to unified schema (required fields present)")
else:
    print("✗ Schema validation failed")

## Metadata Analysis: Hop Depth and Task Type Distribution

Extract and analyze reasoning complexity (hop depth) and task types across datasets.

In [ ]:
# Extract metadata statistics
metadata_stats = {}

for dataset_name, examples in all_examples.items():
    hop_depths = []
    task_types = []
    
    for ex in examples:
        hop_depth = ex.get('metadata_hop_depth', 0)
        task_type = ex.get('metadata_task_type', 'unknown')
        
        hop_depths.append(hop_depth)
        task_types.append(task_type)
    
    metadata_stats[dataset_name] = {
        'hop_depths': hop_depths,
        'task_types': task_types,
        'avg_hop_depth': np.mean(hop_depths) if hop_depths else 0,
        'max_hop_depth': max(hop_depths) if hop_depths else 0,
    }

if VERBOSE:
    for dataset_name, stats in metadata_stats.items():
        print(f"\n{dataset_name}:")
        print(f"  Avg hop depth: {stats['avg_hop_depth']:.1f}")
        print(f"  Max hop depth: {stats['max_hop_depth']}")
        print(f"  Task types: {set(stats['task_types'])}")

## Text Analysis: Input/Output Characteristics

Analyze the complexity of inputs and outputs (text lengths, structure).

In [ ]:
# Analyze input/output text lengths
text_stats = {}

for dataset_name, examples in all_examples.items():
    input_lengths = [len(ex['input']) for ex in examples]
    output_lengths = [len(ex['output']) for ex in examples]
    
    text_stats[dataset_name] = {
        'avg_input_length': np.mean(input_lengths) if input_lengths else 0,
        'max_input_length': max(input_lengths) if input_lengths else 0,
        'avg_output_length': np.mean(output_lengths) if output_lengths else 0,
    }

if VERBOSE:
    for dataset_name, stats in text_stats.items():
        print(f"\n{dataset_name}:")
        print(f"  Avg input length: {stats['avg_input_length']:.0f} chars")
        print(f"  Max input length: {stats['max_input_length']} chars")
        print(f"  Avg output length: {stats['avg_output_length']:.0f} chars")

## Results: Summary and Visualization

Summarize dataset characteristics and visualize key metrics.

In [ ]:
# Build summary DataFrame
summary_data = []

for dataset_name, examples in all_examples.items():
    n_examples = len(examples)
    meta_stats = metadata_stats[dataset_name]
    text_stats_ds = text_stats[dataset_name]
    
    summary_data.append({
        'Dataset': dataset_name,
        'Examples': n_examples,
        'Avg Hop Depth': f"{meta_stats['avg_hop_depth']:.1f}",
        'Max Hop Depth': meta_stats['max_hop_depth'],
        'Avg Input Len': f"{text_stats_ds['avg_input_length']:.0f}",
        'Task Types': ', '.join(set(meta_stats['task_types'])),
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== Dataset Summary ===")
print(summary_df.to_string(index=False))
print()

In [ ]:
# Visualization: Hop depth distribution by dataset
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Hop depth by dataset
datasets = list(metadata_stats.keys())
avg_depths = [metadata_stats[ds]['avg_hop_depth'] for ds in datasets]
max_depths = [metadata_stats[ds]['max_hop_depth'] for ds in datasets]

axes[0].bar(datasets, avg_depths, alpha=0.7, label='Avg', color='steelblue')
axes[0].scatter(datasets, max_depths, color='red', s=100, label='Max', zorder=3)
axes[0].set_ylabel('Hop Depth')
axes[0].set_title('Reasoning Complexity by Dataset')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Input text length by dataset
input_lens = [text_stats[ds]['avg_input_length'] for ds in datasets]
axes[1].bar(datasets, input_lens, alpha=0.7, color='teal')
axes[1].set_ylabel('Avg Characters')
axes[1].set_title('Average Input Length by Dataset')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete.")

## Example Inspection: Sample Examples from Each Dataset

Display complete examples to understand the data structure and reasoning tasks.

In [ ]:
# Display first example from each dataset
for dataset_name, examples in all_examples.items():
    if examples:
        ex = examples[0]
        print(f"\n{'='*70}")
        print(f"DATASET: {dataset_name}")
        print(f"{'='*70}")
        
        # Input (truncated for display)
        inp = ex['input']
        if len(inp) > DISPLAY_INPUT_TRUNCATE_LENGTH:
            inp_display = inp[:DISPLAY_INPUT_TRUNCATE_LENGTH] + "..."
        else:
            inp_display = inp
        print(f"INPUT:\n{inp_display}\n")
        
        # Output
        print(f"OUTPUT: {ex['output']}\n")
        
        # Metadata
        print("METADATA:")
        for key, val in ex.items():
            if key.startswith('metadata_'):
                val_str = str(val)[:100]  # Truncate long metadata values
                print(f"  {key}: {val_str}")

## Summary

This demo successfully loaded and explored the neuro-symbolic reasoning datasets:
- **CLUTRR_v1**: 3 kinship classification examples with multi-hop reasoning chains
- **RuleTaker**: 3 logical entailment examples requiring rule-based inference
- **ProofWriter**: 3 proof question-answering examples with explicit logical structure

All examples conform to the unified schema with standardized input/output format and metadata tracking reasoning complexity (hop depth) and task type. This structure enables systematic evaluation of neuro-symbolic pipelines that convert natural text to first-order logic representations for hybrid reasoning.